# Regresión

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, count, avg, round

spark = SparkSession.builder.appName("Regresion_Computadores").getOrCreate()

ruta_datos = "/home/jovyan/work/semanas/Semana10/modelos/notebooks_etiquetados"
df = spark.read.parquet(ruta_datos)

print(f"Datos cargados: {df.count()} registros")
df.printSchema()
df.show(5, truncate=False)

Datos cargados: 2075 registros
root
 |-- precio_entero: double (nullable = true)
 |-- ram_gb: double (nullable = true)
 |-- almacenamiento_gb: double (nullable = true)
 |-- features: vector (nullable = true)
 |-- scaledFeatures: vector (nullable = true)
 |-- pcaFeatures: vector (nullable = true)
 |-- prediction: integer (nullable = true)

+-------------+------+-----------------+-----------------------+---------------------------------------------------------------+-------------------------------------------+----------+
|precio_entero|ram_gb|almacenamiento_gb|features               |scaledFeatures                                                 |pcaFeatures                                |prediction|
+-------------+------+-----------------+-----------------------+---------------------------------------------------------------+-------------------------------------------+----------+
|888362.0     |16.0  |512.0            |[888362.0,16.0,512.0]  |[-0.3525828816041125,-0.11908881760906984,-

In [11]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Verificar que tenemos las columnas necesarias
print("=== COLUMNAS DISPONIBLES ===")
print(df.columns)

# Seleccionar columnas para regresión
# Vamos a predecir 'precio_entero' usando:
# - ram_gb
# - almacenamiento_gb
# - prediction (cluster del K-Means)

feature_cols = ["ram_gb", "almacenamiento_gb"]

# Si 'marca' está disponible y es numérica, la incluimos
if 'marca' in df.columns and df.schema['marca'].dataType.typeName() in ['double', 'int']:
    feature_cols.append("marca")
    print("Incluyendo 'marca' como feature")

print(f"Features a usar: {feature_cols}")

# 1. Seleccionar solo las columnas necesarias
columnas_necesarias = ["precio_entero"] + feature_cols
df_clean_reg = df.select(columnas_necesarias)

print(f"\nColumnas en df_clean_reg: {df_clean_reg.columns}")

# 2. Vectorizar
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vector = assembler.transform(df_clean_reg)

# 3. Escalar características
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)
df_scaled = scaler.fit(df_vector).transform(df_vector)

# 4. Renombrar variable objetivo
df_scaled = df_scaled.withColumnRenamed("precio_entero", "label_precio")

# 5. Dividir datos
train, test = df_scaled.randomSplit([0.7, 0.3], seed=42)

print(f"\nDatos preparados:")
print(f"Entrenamiento: {train.count()} registros")
print(f"Prueba: {test.count()} registros")

# 6. Entrenar modelo de regresión lineal
lr = LinearRegression(
    featuresCol="scaledFeatures",
    labelCol="label_precio",
    maxIter=10,
    regParam=0.0,
    elasticNetParam=0.0
)

print("\n=== ENTRENANDO REGRESIÓN LINEAL ===")
lr_model = lr.fit(train)

# 7. Predecir
predictions = lr_model.transform(test)

# 8. Mostrar resultados
print("\n=== COMPARATIVA: PRECIO REAL VS PREDICHO ===")
predictions.select("label_precio", "prediction", "ram_gb", "almacenamiento_gb") \
    .show(10, truncate=False)

# 9. Evaluar modelo
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="mae")

r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("\n" + "="*60)
print("     EVALUACIÓN DE LA REGRESIÓN - NOTEBOOKS")
print("="*60)
print(f"R²: {r2 * 100:.2f}%")
print(f"RMSE:      ${rmse:,.0f}")
print("="*60)

# 10. Coeficientes del modelo
print("\n=== COEFICIENTES DEL MODELO ===")
print(f"Intersección (precio base): ${lr_model.intercept:,.0f}")

for i, col_name in enumerate(feature_cols):
    print(f"Coeficiente de '{col_name}': {lr_model.coefficients[i]:.4f}")

# Interpretación
print("\n=== INTERPRETACIÓN ===")
if r2 > 0.7:
    print("Excelente modelo! El precio se explica bien por las variables.")
elif r2 > 0.5:
    print("Modelo aceptable. Hay relación moderada.")
elif r2 > 0.3:
    print("Modelo débil. Considera añadir más variables.")
else:
    print("Modelo muy débil. El precio no se explica bien con estas variables.")

=== COLUMNAS DISPONIBLES ===
['precio_entero', 'ram_gb', 'almacenamiento_gb', 'features', 'scaledFeatures', 'pcaFeatures', 'prediction']
Features a usar: ['ram_gb', 'almacenamiento_gb']

Columnas en df_clean_reg: ['precio_entero', 'ram_gb', 'almacenamiento_gb']

Datos preparados:
Entrenamiento: 1503 registros
Prueba: 572 registros

=== ENTRENANDO REGRESIÓN LINEAL ===

=== COMPARATIVA: PRECIO REAL VS PREDICHO ===
+------------+------------------+------+-----------------+
|label_precio|prediction        |ram_gb|almacenamiento_gb|
+------------+------------------+------+-----------------+
|170879.0    |577425.4686361769 |8.0   |128.0            |
|179910.0    |562512.982546313  |4.0   |120.0            |
|179910.0    |570658.8200231681 |4.0   |127.0            |
|179910.0    |569279.6311593219 |8.0   |121.0            |
|179991.0    |562512.982546313  |4.0   |120.0            |
|181990.0    |427524.81864414294|4.0   |4.0              |
|188443.0    |427524.81864414294|4.0   |4.0          